In [ ]:
import os
import numpy as np
import flopy
import math
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import flopy.utils.postprocessing as pp
from flopy.utils import CellBudgetFile, HeadFile
import matplotlib.colors as mcolors
from unicodedata import name

In [ ]:
import os
import shutil
import numpy as np
import flopy
from scipy.optimize import lsq_linear
import matplotlib.pyplot as plt

##################################
### 1. Global Setup & Cleanup  ###
##################################

dt, nper = 7.0, 52
workspace = "mf6-perimeter-batch"
notebook_dir = os.getcwd()
mf6_exe = os.path.join(notebook_dir, "..", "binaries", "MODFLOW6", "windows", "mf6.exe")

# Close handles and wipe workspace to avoid PermissionErrors
try:
    if 'h_file' in locals(): h_file.close()
    if 'hds_obj' in locals(): hds_obj.close()
except: pass

if os.path.exists(workspace):
    try: shutil.rmtree(workspace)
    except: workspace = f"mf6-perim-v{np.random.randint(100,999)}"
os.makedirs(workspace)

###################################
### 2. Simulation Factory       ###
###################################

def setup_sim(name, wel_data):
    cell_dim, ncol, nrow = 2.0, 50, 50
    lx, ly = ncol * cell_dim, nrow * cell_dim
    sim = flopy.mf6.MFSimulation(sim_name=name, exe_name=mf6_exe, sim_ws=workspace)
    flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=[(dt, 1, 1.0)]*nper)
    ims = flopy.mf6.ModflowIms(sim, complexity="MODERATE", outer_maximum=500)
    gwf = flopy.mf6.ModflowGwf(sim, modelname=name, save_flows=True)
    flopy.mf6.ModflowGwfdis(gwf, nlay=1, nrow=nrow, ncol=ncol, top=0.0, botm=[-10.0],
                            delr=cell_dim, delc=cell_dim, xorigin=-lx/2, yorigin=-ly/2)
    flopy.mf6.ModflowGwfnpf(gwf, k=1e-3, icelltype=0) 
    flopy.mf6.ModflowGwfsto(gwf, ss=1e-5, transient=True)
    flopy.mf6.ModflowGwfic(gwf, strt=0.0)
    flopy.mf6.ModflowGwfwel(gwf, pname='wel', stress_period_data=wel_data)
    flopy.mf6.ModflowGwfoc(gwf, head_filerecord=f"{name}.hds", saverecord=[("HEAD", "ALL")])
    return sim, gwf

######################################
### 3. Generate Target & Candidates ###
######################################

_, dummy_gwf = setup_sim("dummy", {})
grid = dummy_gwf.modelgrid

# A: Compliance Perimeter (The Sensors AND the Candidate Wells)
r_start, r_end, c_start, c_end = 5, 45, 5, 45
cp_cells = []
for c in range(c_start, c_end + 1): cp_cells.extend([(r_start, c), (r_end, c)])
for r in range(r_start + 1, r_end): cp_cells.extend([(r, c_start), (r, c_end)])

candidate_wells = cp_cells # 1:1 Mapping for Perimeter Control
n_cand, n_cp = len(candidate_wells), len(cp_cells)

# B: Random Target Wells (Can be anywhere inside)
wells = [{'r': np.random.randint(r_start+1, r_end), 
          'c': np.random.randint(c_start+1, c_end), 
          'Q': np.random.uniform(0.1, 5, nper)} for _ in range(10)]

wel_spd_target = {p: [((0, w['r'], w['c']), -w['Q'][p]) for w in wells] for p in range(nper)}
sim_t, _ = setup_sim("target", wel_spd_target)
sim_t.write_simulation()
sim_t.run_simulation(silent=True)
hds_t_obj = flopy.utils.binaryfile.HeadFile(os.path.join(workspace, "target.hds"))
hds_t = hds_t_obj.get_alldata()

#######################################
### 4. Build Perimeter Matrix (G)   ###
#######################################

R_full = np.zeros((nper, n_cp, n_cand))
print(f"Characterizing {n_cand} Perimeter Wells...")

for j, (wr, wc) in enumerate(candidate_wells):
    name = f"resp_{j}"
    sim_u, _ = setup_sim(name, {p: [((0, wr, wc), -1.0)] for p in range(nper)})
    sim_u.write_simulation()
    sim_u.run_simulation(silent=True)
    
    h_file = flopy.utils.binaryfile.HeadFile(os.path.join(workspace, f"{name}.hds"))
    h_data = h_file.get_alldata()
    for p in range(nper):
        for i in range(n_cp):
            r_cp, c_cp = cp_cells[i]
            R_full[p, i, j] = 0.0 - h_data[p, 0, r_cp, c_cp]
    h_file.close()

# Assemble Global Matrix G
G = np.zeros((nper * n_cp, nper * n_cand))
b_global = np.array([0.0 - hds_t[p, 0, r, c] for p in range(nper) for r, c in cp_cells])

for pt in range(nper):
    for pp in range(pt + 1):
        dt_idx = pt - pp
        U_inc = R_full[dt_idx, :, :] - (R_full[dt_idx-1, :, :] if dt_idx > 0 else 0)
        G[pt*n_cp:(pt+1)*n_cp, pp*n_cand:(pp+1)*n_cand] = U_inc

#######################################
### 5. Global Solve & Visualization ###
#######################################

alpha = 0.01 # L2 Regularization factor
G_reg = np.vstack([G, np.eye(nper * n_cand) * alpha])
b_reg = np.concatenate([b_global, np.zeros(nper * n_cand)])

print("Solving Global System...")
res = lsq_linear(G_reg, b_reg, bounds=(0, 500))
q_opt = res.x.reshape((nper, n_cand))

# Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
for idx in [0, n_cp//4, n_cp//2]:
    target = np.array([0.0 - hds_t[p, 0, cp_cells[idx][0], cp_cells[idx][1]] for p in range(nper)])
    match = np.zeros(nper)
    for p in range(nper):
        for j in range(p+1):
            dt_idx = p - j
            U_inc = R_full[dt_idx, idx, :] - (R_full[dt_idx-1, idx, :] if dt_idx > 0 else 0)
            match[p] += np.dot(U_inc, q_opt[j, :])
            
    line, = ax1.plot(target, '--', label=f"CP {idx} Target")
    ax1.plot(match, color=line.get_color(), alpha=0.7, label=f"CP {idx} Match")
    ax2.plot(target - match, color=line.get_color(), alpha=0.5)

ax1.set_title("Perimeter-Only Global Batch Optimization")
ax1.legend(); ax1.grid(True, alpha=0.2)
ax2.axhline(0, color='k'); ax2.set_title("Residuals (Error)"); ax2.grid(True, alpha=0.2)
plt.show()

hds_t_obj.close()

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing model target...
    writing model name file...
    writing package dis...
    writing package npf...
    writing package sto...
    writing package ic...
    writing package wel...
INFORMATION: maxbound in ('gwf6', 'wel', 'dimensions') changed to 10 based on size of stress_period_data
    writing package oc...
Characterizing 160 Perimeter Wells...
writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing model resp_0...
    writing model name file...
    writing package dis...
    writing package npf...
    writing package sto...
    writing package ic...
    writing package wel...
INFORMATION: maxbound in ('gwf6', 'wel', 'dimensions') changed to 1 based on size of stress_period_data
    writing package oc...
writing simulation...
  writing simulation name file...
  w

In [ ]:
########################################
### 5. Match and Residual Plots      ###
########################################

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

# Data for residual calculation
all_residuals = []

for idx in [0, n_cp//4, n_cp//2]:
    # Extract Target
    target_curve = np.array([0.0 - hds_r[p, 0, cp_cells[idx], cp_cells[idx]] for p in range(nper)])
    
    # Reconstruct Match (Global)
    match_curve = np.zeros(nper)
    for p in range(nper):
        val = 0
        for j in range(p + 1):
            dt = p - j
            U_inc = R_full[dt, idx, :] - (R_full[dt-1, idx, :] if dt > 0 else 0)
            val += np.dot(U_inc, q_global[j, :])
        match_series[p] = val
    
    # Plot Comparison
    line, = ax1.plot(time_steps, target_curve, '--', label=f"CP {idx} Target")
    ax1.plot(time_steps, match_series, color=line.get_color(), alpha=0.8, label=f"CP {idx} Match")
    
    # Calculate and Plot Residuals
    residuals = target_curve - match_series
    all_residuals.extend(residuals)
    ax2.plot(time_steps, residuals, color=line.get_color(), alpha=0.5)

ax1.set_title("Global Batch Optimization Match")
ax1.set_ylabel("Drawdown (Units)")
ax1.legend()
ax1.grid(True, alpha=0.2)

ax2.axhline(0, color='black', lw=1, linestyle='-')
ax2.set_title("Residuals (Target - Match)")
ax2.set_ylabel("Error")
ax2.set_xlabel("Days")
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Global RMS Error: {np.sqrt(np.mean(np.array(all_residuals)**2)):.4f}")